In [ ]:
import os
import urllib.request
import tensorflow as tf

print("⚡ Fetching your model from GitHub...")

# Define URLs for your uploaded model
urls = [
    "https://raw.githubusercontent.com/payalsolanki26/my-sign-language-model/main/my_asl_model.keras",
    "https://raw.githubusercontent.com/payalsolanki26/my-sign-language-model/main/Models/my_asl_model.keras"
]

model_loaded = False

for url in urls:
    try:
        urllib.request.urlretrieve(url, "model.keras")
        # Check if file downloaded properly (not HTML error)
        if os.path.getsize("model.keras") > 10000:
            model = tf.keras.models.load_model("model.keras")
            model_loaded = True
            print("✅ Successfully loaded `.keras` model!")
            break
    except Exception as e:
        continue

# Fallback: If .keras isn't found, load JSON + Weights
if not model_loaded:
    print("🔄 Loading via JSON + Weights...")
    json_url = "https://raw.githubusercontent.com/payalsolanki26/my-sign-language-model/main/Models/model-bw.json"
    weights_url = "https://raw.githubusercontent.com/payalsolanki26/my-sign-language-model/main/Models/model-bw.weights.h5"

    urllib.request.urlretrieve(json_url, "model.json")
    urllib.request.urlretrieve(weights_url, "model.weights.h5")

    with open("model.json", "r") as f:
        model_json = f.read()
    model = tf.keras.models.model_from_json(model_json)
    model.load_weights("model.weights.h5")
    print("✅ Successfully loaded JSON model weights!")

print("\n🎉 SUCCESS! Model loaded in seconds! Ready for presentation.")

In [ ]:
import time
import cv2
import numpy as np
from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
from base64 import b64decode

# 1. Sign Labels
labels = ['0', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']

# 2. Camera Function
def start_webcam():
    js = Javascript('''
        async function streamWebcam() {
            const div = document.createElement('div');
            const video = document.createElement('video');
            video.style.display = 'block';
            video.style.width = '320px';
            const stream = await navigator.mediaDevices.getUserMedia({video: true});
            document.body.appendChild(div);
            div.appendChild(video);
            video.srcObject = stream;
            await video.play();
            window.captureFrame = function() {
                const canvas = document.createElement('canvas');
                canvas.width = video.videoWidth;
                canvas.height = video.videoHeight;
                canvas.getContext('2d').drawImage(video, 0, 0);
                return canvas.toDataURL('image/jpeg', 0.8);
            };
        }
        streamWebcam();
    ''')
    display(js)

# 3. The Stable Run
start_webcam()
print("🎥 Camera is warming up...")
time.sleep(2)

print("🚀 LIVE PREDICTION RUNNING...")
print("To stop, press the stop [■] button at the top left of this box.\n")

try:
    while True:
        # Get frame
        data = eval_js('captureFrame()')
        if not data: break

        # Decode and Process
        binary = b64decode(data.split(',')[1])
        nparr = np.frombuffer(binary, np.uint8)
        frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        # Resize to model size
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        resized = cv2.resize(gray, (64, 64))
        input_data = np.reshape(resized / 255.0, (1, 64, 64, 1))

        # Predict
        pred = model.predict(input_data, verbose=0)
        idx = np.argmax(pred)
        letter = labels[idx]
        conf = np.max(pred) * 100

        # Output ONLY the prediction clearly
        print(f"\r🖐️ PREDICTED SIGN: [ {letter} ]  (Confidence: {conf:.0f}%)      ", end="")

        # Wait a small bit so it doesn't freeze the browser
        time.sleep(0.5)

except Exception as e:
    print(f"\n🛑 Stream ended.")